# RAG Evaluation Metrics — Exploration

Loads the latest Langfuse export from `local/metric-export/` and produces basic plots.

**Run the export first:**
```bash
python analysis/langfuse_export.py --session-id <your_session_id>
```

In [6]:
import glob
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

EXPORT_DIR = Path("../local/metric-export")


def load_latest(prefix: str = "*langfuse_export*") -> pd.DataFrame:
    files = sorted(EXPORT_DIR.glob(f"{prefix}.parquet"))
    if not files:
        raise FileNotFoundError(f"No parquet files matching {EXPORT_DIR / prefix}")
    path = files[-1]
    print(f"Loading: {path.name}")
    return pd.read_parquet(path)


df = load_latest(prefix="2026-03-04_13-32_langfuse_export_mistral_q4_baseline_001")
print(f"Shape: {df.shape}")
df.head(20)

Loading: 2026-03-04_13-32_langfuse_export_mistral_q4_baseline_001.parquet
Shape: (69, 25)


,observation_id,trace_id,start_time,end_time,latency_ms,run_id,claim_id,metrics_abstention,metrics_recall_at_k,metrics_precision_at_k,...,hardware_swap_out_bytes,score_Answer Correctness,score_Context Recall,score_Correctness,score_Evaluate Hallucination,score_Hallucination based on ground truth,score_abstention,score_mrr,score_precision_at_k,score_recall_at_k
0,29fc7dae68533436,b82a448b188e44069b0f9ca3819dd8bb,2026-03-03 23:01:57.786000+00:00,2026-03-03 23:01:58.762000+00:00,0.976,mistral_q4_baseline_001,emanual_89,None,NaN,NaN,...,NaN,NaN,0.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN
1,0fba773edef6a22d,a71d3133076d3d6a40f4a1b5ca089faa,2026-03-03 23:01:56.712000+00:00,2026-03-03 23:01:57.786000+00:00,1.074,mistral_q4_baseline_001,emanual_420,True,0.000000,0.000000,...,32768.0,NaN,1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
2,a7ca39430afc3b78,261255ec33b880a1617c4d8cc787fa5a,2026-03-03 23:01:56.046000+00:00,2026-03-03 23:01:56.712000+00:00,0.666,mistral_q4_baseline_001,emanual_426,False,0.333333,0.333333,...,0.0,NaN,1.0,0.3,0.3,NaN,NaN,NaN,NaN,NaN
3,0ab8d3686415eeff,5ef945d626a38c937341bcad8c20c9db,2026-03-03 23:01:54.748000+00:00,2026-03-03 23:01:56.046000+00:00,1.298,mistral_q4_baseline_001,emanual_490,True,0.000000,0.000000,...,212992.0,NaN,1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
4,270f9c45db009d1f,b2d3d252542caa811aa99ca8fb499b68,2026-03-03 23:01:52.472000+00:00,2026-03-03 23:01:54.748000+00:00,2.276,mistral_q4_baseline_001,emanual_32,False,0.333333,0.333333,...,16384.0,NaN,0.5,0.4,0.4,NaN,NaN,NaN,NaN,NaN
5,669a3691ef80e9da,ada2a1151d83dc31ce3ad619a7f2e4dd,2026-03-03 23:01:51.615000+00:00,2026-03-03 23:01:52.472000+00:00,0.857,mistral_q4_baseline_001,emanual_413,False,0.333333,0.333333,...,16384.0,NaN,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN
6,4638dd7232084f23,9aae925735d8feee7dcfcfc369eee6a5,2026-03-03 23:01:47.173000+00:00,2026-03-03 23:01:51.615000+00:00,4.442,mistral_q4_baseline_001,emanual_467,False,0.500000,0.333333,...,573440.0,NaN,0.5,0.0,1.0,NaN,NaN,NaN,NaN,NaN
7,17c4e55f8f9cebd7,bdf28bc742c47ceaf983cb24c8ef68c2,2026-03-03 22:34:39.899000+00:00,2026-03-03 22:34:41.478000+00:00,1.579,mistral_q4_baseline_001,emanual_426,None,NaN,NaN,...,NaN,NaN,0.0,0.0,NaN,1.0,NaN,NaN,NaN,NaN
8,bd7c05a44ab89f8d,a8ebddc9feb2550bee8aeb72b4cc7504,2026-03-03 22:34:38.651000+00:00,2026-03-03 22:34:39.899000+00:00,1.248,mistral_q4_baseline_001,emanual_490,True,0.000000,0.000000,...,458752.0,NaN,1.0,0.0,NaN,1.0,NaN,NaN,NaN,NaN
9,e6ed1a08ea1203ad,00c4e620852ae52773f1db7a08a8e41b,2026-03-03 22:34:36.992000+00:00,2026-03-03 22:34:38.651000+00:00,1.659,mistral_q4_baseline_001,emanual_32,True,0.333333,0.333333,...,16384.0,NaN,1.0,0.0,NaN,1.0,NaN,NaN,NaN,NaN


## Summary statistics

In [ ]:
metric_cols = [
    c for c in df.columns
    if any(c.startswith(p) for p in ("metrics_", "latency_", "generation_", "hardware_", "score_"))
]
df[metric_cols].describe().T

## Per-run comparison

In [ ]:
if "run_id" in df.columns:
    agg_cols = [c for c in metric_cols if df[c].dtype in ("float64", "float32", "int64")]
    display(df.groupby("run_id")[agg_cols].mean().round(3))

## Latency distribution

In [ ]:
latency_cols = [c for c in df.columns if "latency" in c]

fig, axes = plt.subplots(1, len(latency_cols), figsize=(5 * len(latency_cols), 4))
if len(latency_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, latency_cols):
    df[col].dropna().hist(bins=30, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("ms")
    ax.set_ylabel("count")

plt.tight_layout()
plt.show()

## Retrieval quality: Recall vs MRR

In [ ]:
recall_col = next((c for c in df.columns if "recall" in c), None)
mrr_col = next((c for c in df.columns if "mrr" in c), None)

if recall_col and mrr_col:
    fig, ax = plt.subplots(figsize=(6, 5))
    groups = df.groupby("run_id") if "run_id" in df.columns else [("all", df)]
    for label, group in groups:
        ax.scatter(group[recall_col], group[mrr_col], label=label, alpha=0.6, s=20)
    ax.set_xlabel(recall_col)
    ax.set_ylabel(mrr_col)
    ax.set_title("Recall@k vs MRR")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("recall or mrr columns not found — skipping scatter plot")

## Langfuse scores (if present)

In [ ]:
score_cols = [c for c in df.columns if c.startswith("score_")]
if score_cols:
    display(df[score_cols].describe().T)
    df[score_cols].hist(bins=20, figsize=(5 * len(score_cols), 4))
    plt.suptitle("Langfuse score distributions")
    plt.tight_layout()
    plt.show()
else:
    print("No score_ columns found — no Langfuse scores were exported")

## Generation throughput

In [ ]:
tps_col = next((c for c in df.columns if "tokens_per_second" in c), None)
if tps_col:
    fig, ax = plt.subplots(figsize=(6, 4))
    if "run_id" in df.columns:
        for label, group in df.groupby("run_id"):
            group[tps_col].dropna().hist(bins=20, ax=ax, alpha=0.7, label=label)
        ax.legend()
    else:
        df[tps_col].dropna().hist(bins=20, ax=ax)
    ax.set_xlabel("tokens/s")
    ax.set_title("Generation throughput")
    plt.tight_layout()
    plt.show()